# 04 — Prepare CNN Windows
# Giai đoạn 1 — Mục 1.5 — Chuẩn bị sliding windows cho CNN raw và envelope
# 
 **Đầu ra**:
- `outputs/tables/windows_cnn_raw.parquet`
- `outputs/tables/windows_cnn_env.parquet`

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

from common import io_utils, dsp, features, config as cfg

In [6]:
import json

OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

manifest_filtered = pd.read_csv(TABLES_DIR / "manifest_filtered.csv")

# Đọc cấu hình bandpass đã chốt ở notebook 02
with open(TABLES_DIR / "bandpass_config.json") as f:
    bandpass_cfg = json.load(f)
BAND_HZ = tuple(bandpass_cfg["band_hz"])
LP_CUTOFF_HZ = bandpass_cfg["lp_cutoff_hz"]

WINDOW_SIZE_RAW = 2048
STRIDE_RAW = 1024
WINDOW_SIZE_ENV = 1024
STRIDE_ENV = 512

# --- Xử lý sampling rate không đồng nhất của Normal baseline (GĐ0 mục 0.1.4) ---
# TODO: nên chuyển mapping này vào common/config.py để tránh lặp lại giữa các notebook
TARGET_FS = 12000
NORMAL_FS_OVERRIDE = {
    "97_Normal_0.mat": 24000,
    "98_Normal_1.mat": 48000,
    "99_Normal_2.mat": 48000,
    "100_Normal_3.mat": 48000,
}

def detect_fs(file_path):
    return NORMAL_FS_OVERRIDE.get(Path(file_path).name, TARGET_FS)

def load_de_signal_fixed_fs(file_path, target_fs=TARGET_FS):
    from scipy.signal import resample_poly
    x = io_utils.load_de_signal(Path(file_path))
    fs = detect_fs(file_path)
    if fs != target_fs:
        x = resample_poly(x, target_fs, fs)
    return x


# Prepare CNN Windows

In [7]:
all_windows_raw = []
all_windows_env = []

for _, row in manifest_filtered.iterrows():
    if row['label'] is None or pd.isna(row['label']):
        continue
    
    # 1. Load signal
    x = io_utils.load_de_signal(Path(row['file_path']))
    file_id = f"{row['label']}_{row['load_hp']}_{row.get('fault_diameter_mils', '')}_{row['file_path']}"
    
    # 2. Xử lý Raw
    w_raw = features.make_sliding_windows(x, WINDOW_SIZE_RAW, overlap_ratio=0.5, file_id=file_id, label=row['label'])
    all_windows_raw.append(w_raw)
    
    # 3. Xử lý Envelope
    envelope = dsp.square_law_envelope(x, fs=12000, band=BAND_HZ, lp_cutoff=500)
    w_env = features.make_sliding_windows(envelope, WINDOW_SIZE_ENV, overlap_ratio=0.5, file_id=file_id, label=row['label'])
    all_windows_env.append(w_env)

In [8]:
windows_raw_df = pd.concat(all_windows_raw, ignore_index=True)
windows_env_df = pd.concat(all_windows_env, ignore_index=True)

windows_raw_df.to_parquet(TABLES_DIR / "windows_cnn_raw.parquet")
windows_env_df.to_parquet(TABLES_DIR / "windows_cnn_env.parquet")
print(f"Raw windows: {len(windows_raw_df)}, Envelope windows: {len(windows_env_df)}")

Raw windows: 5886, Envelope windows: 11832
